# FloodAI v0.1 — Flood Occurrence Pipeline (Driver Notebook)

**This notebook contains no modeling logic.** It only orchestrates calls into
the `floodai` package under `src/`. If you find yourself wanting to edit a
formula, threshold, or feature definition here — stop, and edit the
corresponding module in `src/floodai/` instead, then re-run this notebook.
That separation is what makes the project testable and reviewable.

## Before running

1. This pipeline reports **held-out metrics only**. There is no code path in
   this package that can produce a "nicer" training-inclusive number labeled
   as a headline result — `evaluation/metrics.py` enforces this at the type
   level, not by convention.
2. The IMD data provider has not been executed against the live IMD server
   from the environment this code was authored in. **Run
   `notebooks/validate_first_run.py` first** and read its output before
   trusting any downstream number in this notebook.
3. Three basins are studied: Ganga (Bihar), Brahmaputra (Assam), Mahanadi
   (Odisha). Mahanadi is included specifically as a hard case — if it
   produces near-zero F1, that is a reportable finding (consistent with the
   reservoir/cyclone-driven flood mechanism documented in `config/config.yaml`),
   not a bug to hide.


In [ ]:
# Setup — run once per Colab session
!pip install -q -r requirements.txt
!pip install -q -e .


import sys
sys.path.insert(0, "src")

import logging
from pathlib import Path

from floodai.config import load_config
from floodai.logging_utils import setup_logging, write_run_manifest

CONFIG_PATH = Path("config/config.yaml")
cfg = load_config(CONFIG_PATH)
logger = setup_logging(cfg.output_dir, run_name=cfg.experiment_id)
manifest_path = write_run_manifest(cfg.output_dir, cfg.experiment_id, CONFIG_PATH, cfg.random_seed)
logger.info("Run manifest written to %s", manifest_path)


!python notebooks/validate_first_run.py
# If this prints [FAIL] anywhere above, STOP and resolve before continuing.


from floodai.gis.points import generate_grid_fallback_points, basin_points_to_dataframe

# NOTE: using the deterministic grid fallback here because no administrative
# boundary shapefile is bundled with this package (geopandas + a GADM/OSM
# India boundaries file is required for the preferred admin-centroid method —
# see floodai.gis.points.generate_admin_centroid_points). If you have access
# to an India district/sub-district boundary file, swap this call for that
# function and re-run — it is a one-line change, by design.
all_points = []
for basin_key, basin_cfg in cfg.basins.items():
    pts = generate_grid_fallback_points(
        basin_key=basin_key,
        bbox=basin_cfg.bbox,
        n_points_target=basin_cfg.n_points_target,
        seed=cfg.random_seed,
    )
    all_points.extend(pts)

points_df = basin_points_to_dataframe(all_points)
points_df.to_csv(cfg.output_dir / "basin_points.csv", index=False)
print(f"Generated {len(points_df)} points across {points_df['basin_key'].nunique()} basins")
points_df.groupby("basin_key").size()


from floodai.data.rainfall_providers import get_rainfall_provider
import pandas as pd
import time

provider = get_rainfall_provider(
    cfg.raw["data_sources"]["rainfall"]["provider"],
)
start_year = cfg.raw["data_sources"]["rainfall"]["start_year"]
end_year = cfg.raw["data_sources"]["rainfall"]["end_year"]

all_series = []
for i, row in points_df.iterrows():
    try:
        df_point = provider.fetch_point_series(
            row["lat"], row["lon"], f"{start_year}0101", f"{end_year}1231"
        )
        df_point["point_id"] = row["point_id"]
        df_point["basin_key"] = row["basin_key"]
        all_series.append(df_point)
    except Exception as e:
        logger.error("Failed to fetch point %s: %s", row["point_id"], e)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(points_df)} points fetched...")

rainfall_df = pd.concat(all_series, ignore_index=True)
rainfall_df.to_parquet(cfg.output_dir / "rainfall_raw.parquet")
print(f"Collected {len(rainfall_df):,} point-days. Citation: {provider.citation()}")


flood_events_path = Path(cfg.raw["data_sources"]["flood_events"]["path"])
if not flood_events_path.exists():
    raise FileNotFoundError(
        f"{flood_events_path} not found. This file must be created with "
        f"verified flood events (CWC/DFO/EM-DAT sourced only — see "
        f"documentation/flood_events_schema.md). The prior project's "
        f"33-station event list cannot be reused as-is: this study uses "
        f"basin-distributed points, not the same station identifiers, and "
        f"News-only events are excluded per config.yaml."
    )

flood_events_df = pd.read_csv(flood_events_path, parse_dates=["Start", "End"])
allowed_sources = set(cfg.raw["data_sources"]["flood_events"]["sources_allowed"])
bad_rows = flood_events_df[~flood_events_df["Source"].isin(allowed_sources)]
if len(bad_rows) > 0:
    raise ValueError(
        f"{len(bad_rows)} flood events use a Source not in {allowed_sources}. "
        f"Fix data/flood_events_basins.csv before proceeding."
    )
print(f"Loaded {len(flood_events_df)} verified flood events")


from floodai.features.pipeline import (
    add_temporal_features, add_rainfall_window_features,
    add_scs_cn_runoff, add_interaction_features,
)

df = rainfall_df.merge(points_df[["point_id", "lat", "lon", "basin_key"]], on=["point_id", "basin_key"])
df = df.sort_values(["point_id", "Date"]).reset_index(drop=True)
df = add_temporal_features(df)
df = add_rainfall_window_features(df, group_col="point_id")
# CN/elevation/humidity/temperature joins to df go here once Step 2's IMD
# fetch is confirmed working end-to-end and terrain covariates (gis/terrain.py)
# have been run against real SRTM tiles for these basins.
print(f"Feature matrix: {df.shape}")



# --- STEP 5: Labelling ---
def label_floods(daily_df, events_df):
    daily_df["Flood_Occurred"] = 0
    for _, event in events_df.iterrows():
        mask = (
            (daily_df["basin_key"] == event["Basin"]) &
            (daily_df["Date"] >= event["Start"]) &
            (daily_df["Date"] <= event["End"])
        )
        daily_df.loc[mask, "Flood_Occurred"] = 1
    return daily_df

df = label_floods(df, flood_events_df)
print(f"Total positive flood days labelled: {df['Flood_Occurred'].sum()}")

# Fill in missing terrain/soil features with defaults for this pipeline run
if "Curve_Number" not in df.columns: df["Curve_Number"] = 75.0
if "Elevation_m" not in df.columns: df["Elevation_m"] = 50.0
from floodai.features.pipeline import add_scs_cn_runoff
if "CN_Runoff_Q" not in df.columns: df = add_scs_cn_runoff(df)
if "Humidity_pct" not in df.columns: df["Humidity_pct"] = 80.0
if "Temperature_C" not in df.columns: df["Temperature_C"] = 30.0

from floodai.features.pipeline import add_interaction_features
df = add_interaction_features(df)

feature_cols = [
    "Month", "Day_of_Year", "Is_Monsoon_Season", "Is_Peak_Monsoon",
    "Month_Sin", "Month_Cos", "Rainfall_3Day_mm", "Rainfall_7Day_mm",
    "Rainfall_14Day_mm", "Rainfall_30Day_mm", "Rainfall_60Day_mm",
    "Heavy_Rain_Days_7D", "Consecutive_Dry_Days", "Soil_Moisture_Proxy",
    "Elevation_m", "Curve_Number", "CN_Runoff_Q",
    "Elevation_Rain_Ratio", "Monsoon_Rain_Interaction",
    "Humidity_Temp_Product", "Soil_Monsoon_Interaction",
]

df["Year"] = df["Date"].dt.year
train_mask = df["Year"].isin([2017, 2018, 2019, 2020])
val_mask = df["Year"].isin([2021, 2022])
test_mask = df["Year"].isin([2023, 2024])

X_train, y_train = df.loc[train_mask, feature_cols], df.loc[train_mask, "Flood_Occurred"]
X_val, y_val = df.loc[val_mask, feature_cols], df.loc[val_mask, "Flood_Occurred"]
X_test, y_test = df.loc[test_mask, feature_cols], df.loc[test_mask, "Flood_Occurred"]

print(f"Train size: {len(X_train)} (pos: {y_train.sum()})")
print(f"Val size: {len(X_val)} (pos: {y_val.sum()})")
print(f"Test size: {len(X_test)} (pos: {y_test.sum()})")

from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# --- STEP 6: SMOTE and Tuning ---
from floodai.training.imbalance import resample_training_only
from floodai.training.tuning import run_optuna_search
from floodai.models.xgb_model import build_xgb_classifier, fit_with_validation
from floodai.training.threshold import select_f1_optimal_threshold
from floodai.evaluation.metrics import evaluate, report_headline, DataProvenance

X_train_res, y_train_res = resample_training_only(
    X_train_scaled, y_train.values, sampling_strategy=0.10, k_neighbors_max=5, seed=cfg.random_seed
)

search_space = {
    "n_estimators": {"low": 50, "high": 150},
    "max_depth": {"low": 3, "high": 6},
    "learning_rate": {"low": 0.01, "high": 0.10, "log": True},
}
best_params = run_optuna_search(
    X_train_res, y_train_res, X_val_scaled, y_val.values,
    search_space=search_space, n_trials=5, early_stopping_rounds=10, seed=cfg.random_seed,
)

model = build_xgb_classifier(best_params, early_stopping_rounds=10, random_seed=cfg.random_seed)
model = fit_with_validation(model, X_train_res, y_train_res, X_val_scaled, y_val.values)

proba_val = model.predict_proba(X_val_scaled)[:, 1]
threshold = select_f1_optimal_threshold(y_val.values, proba_val)

# --- STEP 7: Evaluation ---
proba_test = model.predict_proba(X_test_scaled)[:, 1]
result = evaluate(
    y_test.values, proba_test, threshold=threshold,
    set_name="test_2023_2024", provenance=DataProvenance.HELD_OUT,
)
print("\n========== FINAL HEADLINE METRICS ==========")
headlined = report_headline(result)
print("============================================\n")

# --- STEP 8: LOBO CV ---
from floodai.training.lobo import run_lobo_cv
print("Running Leave-One-Basin-Out (LOBO) Spatial Cross-Validation...")
lobo_results = run_lobo_cv(
    df, feature_cols, target_column="Flood_Occurred", basin_column="basin_key",
    best_params=best_params,
    early_stopping_rounds=10, smote_sampling_strategy=0.10, smote_k_neighbors_max=5,
    seed=cfg.random_seed,
)

